# MEMC cvičení 5 – Superpozice a uspořádané výběry

---

Dvě techniky, které z hotových generátorů skládají nové:

1. **Superpozice (směs rozdělení):** hustota je vážený součet jednodušších hustot. Generujeme ve dvou krocích: nejdřív **vybereme složku** (diskrétní úloha ze cvičení 3), pak z ní **vylosujeme hodnotu** (cvičení 4).
2. **Uspořádané výběry:** chceme $n$ rovnoměrných čísel rovnou **seřazených**, a to bez třídění. Ukážeme, že to jde v $O(n)$, a dokonce lze generovat jen **největší hodnoty** z obrovského výběru.

In [ ]:
import math
import random
import time
from bisect import bisect_right
from itertools import accumulate

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def estimate(sampler, n):
    '''Běžné Monte Carlo: průměr n volání generátoru a jeho směrodatná chyba sigma/sqrt(n).'''
    s = s2 = 0.0
    for _ in range(n):
        x = sampler()
        s += x
        s2 += x * x
    mean = s / n
    var = max(s2 / n - mean * mean, 0.0)
    return mean, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    '''Vypíše odhad a ověří, že leží do k směrodatných chyb od přesné hodnoty.'''
    print(f"{name}: {mean:.5f} +- {se:.5f}  (přesně {exact:.5f})")
    assert abs(mean - exact) <= k * se, name


# generátory z předchozích cvičení
def uniform(a, b):
    return a + (b - a) * random.random()


def exponential(lam):
    return -math.log(1.0 - random.random()) / lam


def triangular(a, b, c):
    gamma = random.random()
    hranice = (c - a) / (b - a)   # F(c)
    if gamma < hranice:
        return a + math.sqrt(gamma * (b - a) * (c - a))
    return b - math.sqrt((1.0 - gamma) * (b - a) * (b - c))


def make_bisect(p):
    '''Diskrétní rozdělení s pravděpodobnostmi p: kumulativní součty + půlení intervalu (cvičení 3).'''
    cum = list(accumulate(p))
    return lambda: min(bisect_right(cum, random.random()), len(p) - 1)

## 1. Superpozice (směs rozdělení)

Hustota je vážený součet $m$ jednodušších hustot:

$$f(x) = \sum_{j=1}^m p_j\,f_j(x), \qquad p_j > 0,\quad \sum_j p_j = 1.$$

### Proč to funguje

Označme $J$ diskrétní veličinu s $P(J = j) = p_j$ a $X \mid J = j$ veličinu s hustotou $f_j$. Podle věty o celkové pravděpodobnosti

$$P(X \le x) = \sum_j P(J = j)\,P(X \le x \mid J = j) = \sum_j p_j F_j(x),$$

takže hustota $X$ je přesně $\sum_j p_j f_j$.

### Algoritmus

1. Vylosuj složku $j$ s pravděpodobnostmi $p_j$ (základní algoritmus nebo půlení z cvičení 3).
2. Vylosuj $X$ z $f_j$ (inverzní transformace z cvičení 4).

### Kdy se to hodí

- celková $F^{-1}$ nemá vzorec, ale $F_j^{-1}$ ano
- dvouvrcholová nebo jinak složitá hustota (běžný případ v praxi: „většinou malé, občas velké" škody)
- jedna složka je vzácná, ale důležitá (těžký chvost)

In [ ]:
def make_mixture(vahy, generatory):
    '''f = sum p_j f_j: nejdřív vyber složku j (diskrétně), potom losuj z f_j.'''
    vyber = make_bisect(vahy)
    return lambda: generatory[vyber()]()


vahy = [0.5, 0.3, 0.2]
generatory = [
    lambda: exponential(1.0),          # složka 1: Exp(1), střední hodnota 1
    lambda: uniform(2, 4),             # složka 2: R(2, 4), střední hodnota 3
    lambda: triangular(0, 10, 8),      # složka 3: trojúhelníkové(0, 10; vrchol 8), střední hodnota 6
]

# střední hodnota a rozptyl složek
mu = [1.0, 3.0, (0 + 10 + 8) / 3]
var_slozek = [1.0, (4 - 2) ** 2 / 12, (0 ** 2 + 10 ** 2 + 8 ** 2 - 0 * 10 - 0 * 8 - 10 * 8) / 18]

smes = make_mixture(vahy, generatory)
n = 200_000

# E X = sum p_j mu_j
EX = sum(p * m for p, m in zip(vahy, mu))
check("E X", *estimate(smes, n), EX)

# E X^2 = sum p_j (sigma_j^2 + mu_j^2)  =>  Var X = E X^2 - (E X)^2
EX2 = sum(p * (v + m * m) for p, v, m in zip(vahy, var_slozek, mu))
check("E X^2", *estimate(lambda: smes() ** 2, n), EX2)
print(f"rozptyl směsi: {EX2 - EX ** 2:.4f}")

# P(X < 3) = 0.5 (1 - e^-3) + 0.3 * 0.5 + 0.2 * F_tri(3),  F_tri(x) = x^2 / ((b-a)(c-a)) pro x <= c
f_tri = 3 * 3 / (10 * 8)
presne = 0.5 * (1 - math.exp(-3)) + 0.3 * 0.5 + 0.2 * f_tri
check("P(X < 3)", *estimate(lambda: float(smes() < 3), n), presne)

Střední hodnota směsi je vážený průměr středních hodnot složek. U rozptylu **pozor**: není to vážený průměr rozptylů. Platí

$$D X = \underbrace{\sum_j p_j\,\sigma_j^2}_{\text{uvnitř složek}} + \underbrace{\sum_j p_j\,(\mu_j - \mu)^2}_{\text{mezi složkami}}
= \sum_j p_j(\sigma_j^2 + \mu_j^2) - \mu^2,$$

protože rozptyl roste i tím, že se složky liší středy. Obrázek ukáže hustotu směsi i příspěvky jednotlivých složek:

In [ ]:
def tri_pdf(a, b, c):
    def f(x):
        if x < a or x > b:
            return 0.0
        if x <= c:
            return 2 * (x - a) / ((b - a) * (c - a))
        return 2 * (b - x) / ((b - a) * (b - c))
    return f


hustoty = [
    lambda x: math.exp(-x) if x >= 0 else 0.0,
    lambda x: 0.5 if 2 <= x <= 4 else 0.0,
    tri_pdf(0, 10, 8),
]

xs = [smes() for _ in range(n)]
span, bins = (0, 10), 100
w = (span[1] - span[0]) / bins
grid = [span[0] + (span[1] - span[0]) * i / 1000 for i in range(1001)]

plt.figure(figsize=(8, 4.5))
plt.hist(xs, bins=bins, range=span, weights=[1 / (len(xs) * w)] * len(xs), alpha=0.5, label="histogram ze simulace")
for j, (p, f) in enumerate(zip(vahy, hustoty), start=1):
    plt.plot(grid, [p * f(x) for x in grid], "--", color=["tab:orange", "tab:green", "tab:purple"][j - 1], label=f"$p_{j} f_{j}$")
plt.plot(grid, [sum(p * f(x) for p, f in zip(vahy, hustoty)) for x in grid], "r", label="$f = \\sum p_j f_j$")
plt.xlabel("x")
plt.title("Směs tří rozdělení")
plt.legend()
plt.show()

## 2. Uspořádané výběry

Máme $\gamma_1, \dots, \gamma_n$ nezávislé rovnoměrné na $(0, 1)$ a chceme je seřazené: $\gamma_{(1)} \le \gamma_{(2)} \le \dots \le \gamma_{(n)}$. Naivní cesta je vygenerovat a **setřídit** v $O(n \log n)$. Existuje ale generátor, který **seřazená** čísla vyrábí přímo.

### Odvození

- **Maximum:** $P(\gamma_{(n)} \le x) = P(\text{všech } n \text{ čísel} \le x) = x^n$. Inverzní transformací $\gamma_{(n)} = V_1^{1/n}$ pro $V_1 \sim R(0,1)$.
- **Ostatní zdola:** je-li $\gamma_{(k+1)} = u$, pak zbylých $k$ čísel je (podmíněně) nezávislých rovnoměrných na $(0, u)$. Jejich maximum je $u \cdot V^{1/k}$ podle předchozího bodu, tedy

$$\gamma_{(k)} = \gamma_{(k+1)}\cdot V_{n-k+1}^{1/k}, \qquad k = n-1,\, n-2,\,\dots,\,1.$$

Potřebujeme jen $n$ rovnoměrných čísel a $n$ mocnin: **čas $O(n)$**.

In [ ]:
def ordered_uniforms(n):
    '''Seřazený výběr n rovnoměrných čísel v čase O(n), bez třídění.

    Maximum: U_(n) = V^(1/n); potom U_(k) = U_(k+1) * V^(1/k) směrem dolů.
    '''
    out = [0.0] * n
    u = 1.0
    for k in range(n, 0, -1):
        u *= random.random() ** (1.0 / k)
        out[k - 1] = u
    return out


n_vyb, behy = 10, 100_000

xs = ordered_uniforms(n_vyb)
assert xs == sorted(xs) and 0 <= xs[0] and xs[-1] < 1
print("výstup je uspořádaný:", [round(x, 3) for x in xs])

# E U_(k) = k / (n + 1)
for k in (1, 5, 10):
    m, se = estimate(lambda: ordered_uniforms(n_vyb)[k - 1], behy)
    check(f"E U_({k}), n = {n_vyb}", m, se, k / (n_vyb + 1))

Jak rekurence vypadá: vlevo jeden běh pro $n = 12$. Každý vodorovný pruh je interval $(0, \gamma_{(k)})$ a z něj se losuje další, **menší** číslo; seřazenost vzniká sama. Vpravo je rozdělení maxima $\gamma_{(n)}$ (hustota $n x^{n-1}$) pro $n = 10$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

n12 = 12
us_ = ordered_uniforms(n12)
ax = axes[0]
for k in range(n12, 0, -1):
    ax.barh(k, us_[k - 1], color="tab:blue", alpha=0.25, height=0.6)
    ax.plot([us_[k - 1]], [k], "o", color="tab:red")
ax.set_xlim(0, 1)
ax.set_yticks(range(1, n12 + 1))
ax.set_ylabel("pořadí $k$")
ax.set_xlabel("hodnota")
ax.set_title("$\\gamma_{(n)} = V^{1/n}$, pak $\\gamma_{(k)} = \\gamma_{(k+1)}\\,V^{1/k}$ (seshora dolů)")

maxima = [ordered_uniforms(n_vyb)[-1] for _ in range(50_000)]
w_ = 1 / 40
axes[1].hist(maxima, bins=40, range=(0, 1), weights=[1 / (len(maxima) * w_)] * len(maxima), alpha=0.6, label="max z generátoru")
gr_ = [i / 200 for i in range(201)]
axes[1].plot(gr_, [n_vyb * x ** (n_vyb - 1) for x in gr_], "r", label="hustota $n x^{n-1}$")
axes[1].set_title(f"Rozdělení maxima, $n = {n_vyb}$")
axes[1].legend()
plt.tight_layout()
plt.show()

### Uspořádané exponenciální veličiny

Je-li $F^{-1}$ **rostoucí**, zachovává pořadí. Seřazená čísla z jakéhokoli rozdělení tedy dostaneme transformací seřazených rovnoměrných:

In [ ]:
def ordered_exponentials(n, lam=1.0):
    # F^-1 je rostoucí, takže zachovává pořadí rovnoměrných čísel
    return [-math.log(1.0 - u) / lam for u in ordered_uniforms(n)]


# E[max z n Exp(1)] = H_n = 1 + 1/2 + ... + 1/n  (harmonické číslo)
m, se = estimate(lambda: ordered_exponentials(n_vyb)[-1], behy)
check("E[max z 10 Exp(1)]", m, se, sum(1 / j for j in range(1, n_vyb + 1)))

# E[min z n Exp(1)] = 1/n  (minimum z n exponenciálních je Exp(n))
m, se = estimate(lambda: ordered_exponentials(n_vyb)[0], behy)
check("E[min z 10 Exp(1)]", m, se, 1 / n_vyb)

# obecně E X_(k) = 1/n + 1/(n-1) + ... + 1/(n-k+1)  (Rényiho reprezentace)
k = 5
m, se = estimate(lambda: ordered_exponentials(n_vyb)[k - 1], behy)
check(f"E X_({k}), n = {n_vyb}", m, se, sum(1 / (n_vyb - i) for i in range(k)))

### Jen největší hodnoty z obrovského výběru

Právě rekurence „shora dolů" umožňuje něco, co třídění neumí: **vygenerovat jen $k$ největších čísel z $n$**, aniž bychom generovali ostatní. Stačí $k$ kroků místo $n$.

Praktický příklad: největší škoda z $n = 10^8$ pojistných událostí. Naivně bychom museli vygenerovat a uchovat sto milionů čísel.

In [ ]:
def nejvetsi_rovnomerne(n, k):
    '''k největších z n rovnoměrných čísel, sestupně, v čase O(k) bez ohledu na n.'''
    u = 1.0
    out = []
    for i in range(k):
        u *= random.random() ** (1.0 / (n - i))
        out.append(u)
    return out


N = 10 ** 8
t0 = time.perf_counter()
top = nejvetsi_rovnomerne(N, 3)
print(f"3 největší z {N:,} rovnoměrných čísel (za {(time.perf_counter() - t0) * 1e6:.0f} us):".replace(",", " "))
print("  ", top)

# Čísla jsou skoro u 1, takže se podíváme na N * (1 - U_(N-j)) ~ Gamma(j + 1) se střední hodnotou (j + 1) * N/(N + 1)
for j in range(3):
    m, se = estimate(lambda: N * (1 - nejvetsi_rovnomerne(N, j + 1)[j]), 100_000)
    check(f"E[N (1 - U_(N-{j}))]", m, se, (j + 1) * N / (N + 1))

Kontrola je na škále $N(1 - \gamma)$, protože samotná čísla jsou od jedničky vzdálená jen o $\sim 10^{-8}$.

### Rychlost: je to opravdu rychlejší než třídění?

Asymptoticky ano ($O(n)$ proti $O(n \log n)$), ale v čistém Pythonu rozhodují i konstanty: standardní `sorted` je napsaný v C a dělá to velmi rychle. Změřme:

Existuje i druhá cesta bez třídění: jsou-li $S_k = E_1 + \dots + E_k$ částečné součty nezávislých $\operatorname{Exp}(1)$, pak $\big(S_1/S_{n+1}, \dots, S_n/S_{n+1}\big)$ má stejné rozdělení jako seřazený výběr $n$ rovnoměrných čísel. (Za tím stojí souvislost s Poissonovým procesem: body procesu na intervalu jsou, při daném počtu, seřazená rovnoměrná čísla.)

In [ ]:
def ordered_uniforms_cumsum(n):
    '''Druhá cesta: částečné součty exponenciálních mezer, vydělené posledním.'''
    s = list(accumulate(-math.log(1.0 - random.random()) for _ in range(n + 1)))
    posledni = s[-1]
    return [x / posledni for x in s[:-1]]


# správnost: E U_(k) = k / (n + 1)
m, se = estimate(lambda: ordered_uniforms_cumsum(n_vyb)[4], behy)
check("E U_(5), cumsum, n = 10", m, se, 5 / (n_vyb + 1))


def cas(f, opakovani=5):
    nejlepsi = float("inf")
    for _ in range(opakovani):
        t0 = time.perf_counter()
        f()
        nejlepsi = min(nejlepsi, time.perf_counter() - t0)
    return nejlepsi


print(f"\n{'n':>9s} {'sorted':>10s} {'ordered':>10s} {'cumsum':>10s}   (čas v milisekundách, nejlepší z 5)")
for nn in (1_000, 100_000):
    t_sort = cas(lambda: sorted(random.random() for _ in range(nn)))
    t_ord = cas(lambda: ordered_uniforms(nn))
    t_cum = cas(lambda: ordered_uniforms_cumsum(nn))
    print(f"{nn:9,d} {t_sort * 1e3:10.2f} {t_ord * 1e3:10.2f} {t_cum * 1e3:10.2f}".replace(",", " "))

Na našem stroji je $O(n)$ metoda zhruba **1.5–2× rychlejší** než `sorted`, a to už pro $n = 1000$. Zrychlení není dramatické, protože `sorted` je napsaný v C (přesná čísla závisí na počítači). **Skutečná výhoda** ale není rychlost třídění, nýbrž to, že

- umíme generovat **jen největší (nebo nejmenší) hodnoty** v čase $O(k)$,
- umíme generovat **postupně** (po jednom, seřazeně), aniž bychom čekali na celý výběr,
- tytéž myšlenky se použijí u procesů, kde potřebujeme **okamžiky událostí**.

### Použití: okamžiky požadavků v pojišťovně

> Tohle je přesně ten díl, který ve **cvičení 9** dovolí nakreslit průběh roku: počet požadavků je binomický a jejich okamžiky jsou seřazený rovnoměrný výběr.

V modelu z cvičení 2 je počet požadavků za rok zhruba Poissonův s $\lambda = Np = 18.25$. Při daném počtu $K$ jsou okamžiky požadavků **seřazená rovnoměrná čísla** na $[0, 1]$ (roku). Můžeme tedy generovat celý časový průběh roku bez procházení 365 dnů.

In [ ]:
def poisson(lam):
    '''Poissonovo rozdělení násobením rovnoměrných čísel (cvičení 3).'''
    q = math.exp(-lam)
    n = 1
    s = random.random()
    while s > q:
        s *= random.random()
        n += 1
    return n - 1


def okamziky_pozadavku(lam):
    '''Okamžiky požadavků v roce [0, 1]: Poissonův počet seřazených rovnoměrných čísel.'''
    return ordered_uniforms(poisson(lam))


lam = 365 * 0.05
print("příklad jednoho roku:", [round(t, 2) for t in okamziky_pozadavku(lam)][:8], "...")

# E (počet požadavků v první polovině roku) = lam / 2
m, se = estimate(lambda: float(sum(t < 0.5 for t in okamziky_pozadavku(lam))), 50_000)
check("E počet požadavků v 1. polovině roku", m, se, lam / 2)

## 3. Alternativa: NumPy

NumPy má setřídění (`np.sort`) i vektorové operace. Obě metody bez třídění se vektorově zapíšou jednoduše. Rychlostně jsou si s `np.sort` blízké, protože i ten je velmi dobře optimalizovaný:

In [ ]:
try:
    import numpy as np

    rng = np.random.default_rng(1)
    N = 1_000_000

    t0 = time.perf_counter()
    a = np.sort(rng.random(N))
    t_sort = time.perf_counter() - t0

    t0 = time.perf_counter()
    s = np.cumsum(rng.exponential(size=N + 1))       # částečné součty exponenciálních mezer
    b = s[:-1] / s[-1]                               # = seřazený výběr
    t_cum = time.perf_counter() - t0

    print(f"np.sort:   {t_sort * 1e3:6.1f} ms,  E U_(k)/(k/(N+1)) pro k = N/2: {a[N // 2 - 1] / (N // 2 / (N + 1)):.4f}")
    print(f"cumsum:    {t_cum * 1e3:6.1f} ms,  E U_(k)/(k/(N+1)) pro k = N/2: {b[N // 2 - 1] / (N // 2 / (N + 1)):.4f}")
    assert (np.diff(b) >= 0).all()

    # směs vektorově: vybereme složky najednou a pak losujeme z každé
    j = rng.choice(3, size=N, p=[0.5, 0.3, 0.2])
    x = np.where(j == 0, rng.exponential(1.0, N),
        np.where(j == 1, rng.uniform(2, 4, N), rng.triangular(0, 8, 10, N)))
    print(f"směs vektorově: E X = {x.mean():.4f}  (přesně 2.6)")
except ModuleNotFoundError:
    print("NumPy není nainstalované, spusťte:  uv sync")

Pozor na pořadí parametrů u NumPy `triangular(left, mode, right)`: náš vrchol $c = 8$ je tam prostřední.

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Superpozice | $f = \sum p_j f_j$; nejdřív vylosuj složku $j$ (diskrétně), potom hodnotu z $f_j$ |
| Střední hodnota směsi | $\sum p_j\mu_j$ |
| Rozptyl směsi | $\sum p_j(\sigma_j^2 + \mu_j^2) - \mu^2$, **není** vážený průměr rozptylů |
| Seřazený výběr | $\gamma_{(n)} = V^{1/n}$, $\gamma_{(k)} = \gamma_{(k+1)}V^{1/k}$; čas $O(n)$ bez třídění |
| Jen top $k$ z $n$ | stačí $k$ kroků, nezáleží na $n$ |
| Jiné rozdělení | transformace rostoucí $F^{-1}$ zachová pořadí |
| Rychlost | $O(n)$ metoda je jen asi 2× rychlejší než `sorted` (ten je v C); skutečná výhoda je top $k$ a postupné generování |

### Zkuste si

1. Vytvořte **směs dvou normálních** rozdělení $N(-2, 1)$ a $N(3, 0.5)$ s váhami $0.4$ a $0.6$ (použijte `random.gauss`). Ověřte střední hodnotu a rozptyl vzorci z textu a nakreslete histogram. Je rozdělení dvouvrcholové?
2. **Škody ve dvou třídách:** s pravděpodobností $0.9$ je škoda $\operatorname{Exp}$ se střední hodnotou $1000$, s pravděpodobností $0.1$ je to $\operatorname{Exp}$ se střední hodnotou $20\,000$. Spočítejte střední hodnotu a rozptyl směsi a vysvětlete, proč směs vypadá „těžkoocasě" (kolik procent rozptylu pochází z velkých škod?).
3. Pomocí `nejvetsi_rovnomerne` vygenerujte **největší škodu** z $n = 10^6$ pojistných událostí s exponenciální výší $\operatorname{Exp}(1)$ (transformujte $-\ln(1 - u)$). Ověřte střední hodnotu $H_n \approx \ln n + 0.5772$.